In [ ]:
import moku_util as mu

import scipy.optimize as optimize

# The moku_util module loads these libraries for internal methods,
# but you may want to run alternative code that uses them directly.
import matplotlib.pyplot as plt
import numpy as np

#### Load the data

In [ ]:
# Acquired data from both 'slow' channels of the SiPM readout, to serve
# as example. You'll probably want to acquire your own data and change
# the filename appropriately.
#
# Note that the file itself is too large to be hosted on GitHub, but 
# is available on Canvas for download. You shouldn't need to use it
# since you can acquire your own data, but it's there.
filename = '../example_data/cs137_2channel_pulses.li'
df = mu.DataFile(filename)

#### Find the pulses in the data and integrate them, for both channels

Make sure to change **all the function parameters** to match the values you determined in the previous notebook

In [ ]:
height = 0.5
width = 2
prominence = (0.3, None)
pre_peak = 3
post_peak = 8

for channel in [0, 1]:
    df.find_peaks(
        channel=channel, 
        height=height, 
        width=width, 
        prominence=prominence,
        negative_signal=False
    )

    df.integrate_peaks(
        channel=channel, 
        pre_peak=pre_peak, 
        post_peak=post_peak
    )

#### Identify the spectral peak associated to the full-energy deposition

First, we'll make the same histogram as before (adjust `bin_edges` etc to match what you were using before). We'll focus on one channel for now, then extend our analysis to both channels.

In [ ]:
# Set the number of bins for a histogram of the pulse integrals.
# This will be the primary parameter you can adjust, although you 
# may also want to generate a bin_edges array manually once you
# know what's happening.
nbins = 100

# Set the bin_edges dynamically using the max and min of the 
# pulse integrals and the number of bins you specified
bin_edges = np.linspace(
    df.peak_integrals[0].min(), 
    df.peak_integrals[0].max(), 
    nbins+1
)

# Alternatively, you could use logarithmic spacing for the bins,
# but leave this commented out for now
# bin_edges = np.logspace(
#     np.log10(df.peak_integrals[0].min()), 
#     np.log10(df.peak_integrals[0].max()), 
#     nbins+1
# )

# Plot the data and extract the histogram values in one go
%matplotlib widget
fig, ax = plt.subplots(figsize=(6,4))
vals1, _, _ = ax.hist(df.peak_integrals[0], bins=bin_edges)
ax.set_xlabel('Pulse Integral (arb)')
ax.set_ylabel('Number of events per bin')
# ax.set_yscale('log') # You may wish to use a logarithmic scale
# ax.set_xscale('log')
fig.tight_layout()
plt.show()

In [ ]:
# Close the figure to avoid memory leaks with the widget backend
plt.close('all')

Examine your spectrum and identify the lower and upper limits of the full-energy _photopeak_ from Cs-137. We'll use these values to select only the points that are obviously from the full energy peak so we can fit those points to a model.

In [ ]:
# Record your upper and lower limits for the peak of interest.
lower_limit = None
upper_limit = None

# Now, calculate the midpoints of each bin, which we interpret
# as the energy/pulse area values for each bin.
pulse_areas = (bin_edges[:-1] + bin_edges[1:])/2

# Build a mask select on the bins that fall within the specified
# limits, then apply that mask to the data
mask = (pulse_areas >= lower_limit) & (pulse_areas <= upper_limit)
hist_fit_xvals = pulse_areas[mask]
hist_fit_yvals = vals1[mask]

#### Let's fit the full-energy photopeak

In a world without any statistical fluctuations, a mono-energetic gamma ray would produce a mono-energetic pulse of charge, which would yield the same integral every time. This would result in a delta-function in the energy spectrum. Clearly, this is not what we observe.

A variety of statistical fluctuations contribute to the _line-width_ of the full-energy peak: fluctuations in the number of scintillation photons produced for each gamma ray interaction, fluctuations in the number of photons "counted" by the SiPM (hint: counted), as well as electronic fluctuations in the gain of the SiPM and the readout electronics. The first two are easily quantifiable, while the last one is more difficult and out of the scope of this lab, but will be sub-dominant by design.

A full discussion of the expected shape of the full-energy is also out of scope, but can be roughly approximated by a Gaussian distribution. We'll fit our data to a Gaussian and try to interpret the result. The fit includes a constant offset to account for background.

In [ ]:
# Define a function to fit to the data. This is a Gaussian plus a constant
def gaussian_plus_constant(x, A, mu, sigma, C):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + C

In [ ]:
# Examine the spectrum and make some initial guesses for the fit parameters.
# This is done automatically by analyzing the masked data, but you may need
# to adjust these guesses to get the fit to converge
A_guess = hist_fit_yvals.max() - hist_fit_yvals.min()
mu_guess = hist_fit_xvals[np.argmax(hist_fit_yvals)]
sigma_guess = 0.25*(hist_fit_xvals.max() - hist_fit_xvals.min())
C_guess = hist_fit_yvals.min()

p0 = [A_guess, mu_guess, sigma_guess, C_guess]

In [ ]:
# Do a rough check of your initial guesses by plotting the function with
# the initial parameters on top of the data
x_plot = np.linspace(hist_fit_xvals.min(), hist_fit_xvals.max(), 1000)
y_guess = gaussian_plus_constant(x_plot, *p0)

%matplotlib inline
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(hist_fit_xvals, hist_fit_yvals, 'o', label='Data')
ax.plot(x_plot, y_guess, label='Initial Guess')
ax.set_xlabel('Pulse Integral (arb)')
ax.set_ylabel('Number of events per bin')
ax.legend(fontsize=10)
plt.show()

If the above looks kind of close, proceed with the notebook. If not, adjust the initial parameters appropriately.

In [ ]:
# Fit the data using scipy's curve_fit function
popt, pcov = \
    optimize.curve_fit(
        gaussian_plus_constant, 
        hist_fit_xvals, 
        hist_fit_yvals, 
        p0=p0
    )

#### Let's plot the fit result back on our histogram as a final check!

In [ ]:
%matplotlib widget
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(df.peak_integrals[0], bins=bin_edges)
ax.plot(x_plot, gaussian_plus_constant(x_plot, *popt), ls='--', lw=3, color='red')
ax.set_xlabel('Pulse Integral (arb)')
ax.set_ylabel('Number of events per bin')
# ax.set_yscale('log') # You may wish to use a logarithmic scale
# ax.set_xscale('log')
fig.tight_layout()
plt.show()

In [ ]:
# Close the figure to avoid memory leaks with the widget backend
plt.close('all')

---
#### With a good fit in hand, let's interpret our results

Even if the Gaussian shape isn't exactly right, it's a reasonable guess from which we can extract two important parameters:
1. From the mean of the Gaussian, we can estimate the proportionality factor to convert from pulse area to gamma ray energy using the known energy of the primary emission from Cs-137.
2. From the standard deviation/width of the Gaussian, we can estimate the scale of the dominant fluctuations

For now, let's think about fluctuations

In [ ]:
mu = popt[1]
sigma = popt[2]

print('Standard deviation in arbitrary units : ', sigma)
print('         Relative uncertainty in peak : ', sigma/mu)

Examine the relative uncertainty. Is it consistent with the reported energy resolution of the scintillator? A value of energy resolution specific to your scintillator module is printed on the side of the metal housing!

If it's not consistent with this energy resolution, it's probably due to limited _counting statistics_ associated with photon collection. If we don't have enough photons, Poisson-like fluctuations will dominate! We can use this fact to roughly estimate how many photons the SiPM collects per full-energy deposition, since $\mu \propto N_{\rm photons}$.

To be complete though, note that the energy resolution of the scintillator will contribute to the standard deviation that we measure, so we'll have to take that into account simply by noting that variance is additive, i.e.

$$ \sigma_{\rm meas}^2 \approx \sigma_{\rm Poisson}^2 + \sigma_{\rm scint}^2 $$

Remembering that $\sigma_{\rm Poisson} = \sqrt{N}$, an estimate of $N_{\rm photons}$ can be found via,

$$ N_{\rm photons} \approx \frac{\mu^2}{\sigma_{\rm meas}^2 - \sigma_{\rm scint}^2}$$

with $\sigma_{\rm scint} \approx (\text{scintillator resolution}) \cdot \mu$.

In [ ]:
# Define your scintillator resolution by examining the label on the metal
# housing of your scintillator. This is often given as a percentage, which
# you can convert to a relative uncertainty by dividing by 100.
sigma_scint = None

N_photons = mu**2 / (sigma**2 - (sigma_scint*mu)**2)

print('Estimated number of photons detected : ', N_photons)

---
#### Comparing the two channels and accommodating differential gain

So why do we have two SiPMs in the first place? We can improve the energy resolution of our apparatus by collecting more photons per event, given that Poisson fluctuations are contributing significantly. Two SiPMs is twice the photons!

However, we have to be careful about doing this right, since the SiPMs themselves may have different gains. Each device has a unique breakdown voltage, but since we're operating both with the same bias voltage, the amount of charge/current that each produces per detected photon will differ.

Thankfully, the known energy of the Cs-137 gamma ray will allow us to scale the detected energies (read: pulse integrals) from one SiPM to the other.

#### Let's first look at spectra from both channels, then we'll fit the second channel in the same way as the first

In [ ]:
# Examine both spectra together first
%matplotlib widget
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(df.peak_integrals[0], bins=bin_edges, alpha=0.5, label='Input 1')
ax.hist(df.peak_integrals[1], bins=bin_edges, alpha=0.5, label='Input 2')
ax.set_xlabel('Pulse Integral (arb)')
ax.set_ylabel('Number of events per bin')
ax.legend(fontsize=10)
fig.tight_layout()
plt.show()

#### Spend a moment examining this figure

What's different between the two inputs? What's similar? Are your observations consistent with what you would expect for differential gain?

In [ ]:
# Close the figure to avoid memory leaks with the widget backend
plt.close('all')

#### Identify upper and lower limits for the second channel, then perform the fit

In [ ]:
# Enter your values here for the limits of the peak in the second input
lower_limit2 = None
upper_limit2 = None

# Build a mask to select the bins that fall within the specified range
mask2 = (pulse_areas >= lower_limit2) & (pulse_areas <= upper_limit2)

# Build the histogram data 
vals2, _ = np.histogram(df.peak_integrals[1], bins=bin_edges)

# Get the x and y values for the fit by applying the mask
hist_fit_xvals2 = pulse_areas[mask2]
hist_fit_yvals2 = vals2[mask2]

# Build an initial guess for the second channel, using the same method
A_guess2 = hist_fit_yvals2.max() - hist_fit_yvals2.min()
mu_guess2 = hist_fit_xvals2[np.argmax(hist_fit_yvals2)]
sigma_guess2 = 0.25*(hist_fit_xvals2.max() - hist_fit_xvals2.min())
C_guess2 = hist_fit_yvals2.min()
p0_2 = [A_guess2, mu_guess2, sigma_guess2, C_guess2]

# Fit the second channel data
popt2, pcov2 = \
    optimize.curve_fit(
        gaussian_plus_constant, 
        hist_fit_xvals2, 
        hist_fit_yvals2, 
        p0=p0_2
    )

In [ ]:
# Plot the fit result on top of the data for the second channel 
# to check that our fit worked
x_plot2 = np.linspace(hist_fit_xvals2.min(), hist_fit_xvals2.max(), 1000)

%matplotlib widget
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(df.peak_integrals[1], bins=bin_edges)
ax.plot(x_plot2, gaussian_plus_constant(x_plot2, *popt2), ls='--', lw=3, color='red')
ax.set_xlabel('Pulse Integral (arb)')
ax.set_ylabel('Number of events per bin')
fig.tight_layout()
plt.show()

In [ ]:
# Close the figure to avoid memory leaks with the widget backend
plt.close('all')

#### So now we have fits of both of the uncalibrated spectra!

There are many different ways you could imagine using these fits to account for the differential gain, but we'll proceed simply by normalizing each of the pulse integrals by the measured mean of the full-energy peak, so that the full-energy peak would be centered around 1 in each spectrum.

However, we can't just add the two histograms after doing this. We have to add the pulse areas _event by event_. This is complicated by the fact that each SiPM may individually see different spurious events, so we have loop over every event captured by one channel, and look for the corresponding event in the other channel, and then add their areas.

Without going too much into the gory details, this is done for you below.

In [ ]:
# The "peaks" attribute of the DataFile contains the indices (i.e. times)
# of the peaks that were found in each channel. We'll loop over one channel
# and check for peaks in the other channel that occur in some short time
# window around the peaks in the first channel. If we find any, we'll record
# the integrals of those peaks in a new list.
peaks1 = df.peaks[0]
peaks2 = df.peaks[1]
peak_int1 = df.peak_integrals[0]
peak_int2 = df.peak_integrals[1]

# Define the mean value of the full-energy peak for each channel
# for easy reference and normalization.
mu1 = popt[1]
mu2 = popt2[1]

# Define a time window, in terms of the number of samples. This should
# be shorter than the total number of samples per pulse
window = 5

coincident_peak_integrals = []
for idx, peak in enumerate(peaks1):
    # Find peaks in the second channel within the time window
    mask = (peaks2 >= peak - window) & (peaks2 <= peak + window)
    coincident_peaks = peaks2[mask]

    # If any coincident peaks are found, record their integrals
    # In case of pulse pileup, we may find more than one peak in
    # the second channel, so we'll just ignore those events and 
    # only count coincident peaks when there is clearly only one
    # peak per channel in the specified window.
    if len(coincident_peaks) == 1:
        coincident_peak_integrals.append(peak_int1[idx]/mu1 + peak_int2[mask][0]/mu2)
    else:
        continue

# Since each pulse was normalized to around 1 individually, their sum
# would be centered around 2. We'll just divide by 2 to get the 
# combined pulse area back to around 1 for easier comparison with the
# individual channels.
coincident_peak_integrals = np.array(coincident_peak_integrals)/2

#### Finally, let's plot the spectrum of coincident peaks, together with each channel individually

In [ ]:
# We need to start by defining some new bin edges.
# You may want to manually set these after observing the
# distribution of the coincident peak integrals, but we'll just
# set them dynamically here based on the max and min of the data.
nbins = 100
coincident_bin_edges = np.linspace(
    coincident_peak_integrals.min(), 
    coincident_peak_integrals.max(), 
    nbins+1
)

# Alternative logarithmic binning, if you want to try that
# coincident_bin_edges = np.logspace(
#     np.log10(coincident_peak_integrals.min()),
#     np.log10(coincident_peak_integrals.max()),
#     nbins+1
# )

%matplotlib widget
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(coincident_peak_integrals, bins=coincident_bin_edges, label='Coincident sum', density=True)
ax.hist(df.peak_integrals[0]/mu1, bins=coincident_bin_edges, alpha=0.5, label='Input 1', density=True)
ax.hist(df.peak_integrals[1]/mu2, bins=coincident_bin_edges, alpha=0.5, label='Input 2', density=True)
ax.set_xlabel('Sum of Pulse Integrals (arb)')
ax.set_ylabel('Number of events per bin')
ax.legend(fontsize=10)
# ax.set_yscale('log') # You may wish to use a logarithmic scale
# ax.set_xscale('log')
fig.tight_layout()
plt.show()

In [ ]:
# Close the figure to avoid memory leaks with the widget backend
plt.close('all')

#### Examine the combined histograms

What is different between the individual spectra and the coincident spectrum?

Can you explain these differences conceptually?

If you're feeling up to it, you could try reproducing some of the fitting code to extract the apparent width of the full-energy peak in the coincident spectrum, and then re-estimate the number of photons via the same method. What would you expect to find from this analysis?